# DWBI Week 4 — Dimensional Modeling
**Ali Raza** · `su92-bsdsm-f23-018` · **BSDS-7A**

PakFreight's Week-3 star (1 fact + 5 dims, 34 rows, 578,650 kg) faces the snowflake question: should DimCity be normalized? Seven tasks count the joins, build the snowflake, do the storage arithmetic, register the relationships, settle cross-filter direction — and the totals do not move by a single kilogram.

## T3 — Storage arithmetic (12 marks)

The storage argument is the snowflake's only real win. Work out how big that win actually is.

**The model:** 2 bytes per character (the way Excel and a columnar dictionary hold text), 4 bytes per integer key.

### Part A — the three ways to hold it

| Rows in DimCity | Flat, raw (bytes) | Flat, encoded (bytes) | Snowflaked (bytes) | Saving (encoded − snow) |
|---|---|---|---|---|
| 15 | 412 | 274 | 292 | **−18** |
| 1,000 | 27,467 | 8,154 | 4,232 | 3,922 |
| 100,000 | 2,746,667 | 800,154 | 400,232 | 399,922 |
| 5,000,000 | 137,333,333 | 40,000,154 | 20,000,232 | 19,999,922 |
| 50,000,000 | 1,373,333,333 | 400,000,154 | 200,000,232 | 199,999,922 |

- **Flat, raw:** ProvinceName + RegionName written out as text on every city row (412 bytes at 15 rows).
- **Flat, encoded:** `154 + 8N` — the dictionary holds each distinct string once (154 bytes), plus two 4-byte pointers per row.
- **Snowflaked:** `232 + 4N` — DimProvince itself (232 bytes fixed) plus one 4-byte ProvinceKey per city row.
- **Saving(N) = 4N − 78.**

![Storage across row counts](charts/t3_storage.png)

### Part B — the finding

At 15 rows the snowflake is **BIGGER by 18 bytes** — DimProvince has to be paid for before it saves
anything. Once encoded, the saving per city row is exactly **4 bytes**: two pointers become one.
That is the entire mechanism.

### Part C — put it in context

A fact table runs ~100× its dimensions at ~40 bytes/row:

| Rows in DimCity | Saving (bytes) | Fact bytes | Saving as % of fact |
|---|---|---|---|
| 15 | −18 | 60,000 | −0.03% |
| 50,000,000 | 199,999,922 | 200,000,000,000 | ≈0.1% |

**The one condition** under which the storage argument is worth paying for: **wide, high-cardinality
text that will not dictionary-compress, on a large dimension** — or any row store where the dictionary
does not exist.

In [ ]:
# T3 verified in code
for N in [15, 1000, 100000, 5000000, 50000000]:
    enc, snf = 154 + 8*N, 232 + 4*N
    print(f"N={N:>10,}: encoded={enc:>12,}  snowflaked={snf:>12,}  saving={enc-snf:>12,}")
assert (154+8*15)-(232+4*15) == -18, "the trap row"
print("\nTrap confirmed: at 15 rows the snowflake is BIGGER by 18 bytes")
print(f"Part C: 199999922/200000000000 = {199999922/200000000000*100:.4f}% ≈ 0.1%")